![Credit card being held in hand](credit_card.jpg)

Commercial banks receive _a lot_ of applications for credit cards. Many of them get rejected for many reasons, like high loan balances, low income levels, or too many inquiries on an individual's credit report, for example. Manually analyzing these applications is mundane, error-prone, and time-consuming (and time is money!). Luckily, this task can be automated with the power of machine learning and pretty much every commercial bank does so nowadays. In this workbook, you will build an automatic credit card approval predictor using machine learning techniques, just like real banks do.

### The Data

The data is a small subset of the Credit Card Approval dataset from the UCI Machine Learning Repository showing the credit card applications a bank receives. This dataset has been loaded as a `pandas` DataFrame called `cc_apps`. The last column in the dataset is the target value.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Load the dataset
cc_apps = pd.read_csv("cc_approvals.data", header=None)
cc_apps.head()

# Preprocess data
cc_apps = cc_apps.replace("?", np.nan)
X = cc_apps.iloc[:, :-1].copy()
y = cc_apps.iloc[:, -1].map({"+": 1, "-": 0})

for col in X.columns:
    try:
        X[col] = pd.to_numeric(X[col])
    except Exception:
        pass

for col in X.columns:
    if X[col].dtype == "O":
        X[col] = X[col].fillna(X[col].mode()[0])
    else:
        X[col] = X[col].fillna(X[col].median())

X = pd.get_dummies(X, drop_first=True)
X.columns = X.columns.astype(str)

# Train and evaluate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

grid = GridSearchCV(
    LogisticRegression(max_iter=3000),
    {"C": [0.01, 0.1, 1, 10]},
    cv=5,
)
grid.fit(X_train_scaled, y_train)

y_pred = grid.predict(X_test_scaled)
best_score = float(accuracy_score(y_test, y_pred))

print("Best params:", grid.best_params_)
print("best_score:", round(best_score, 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("Target met (>= 0.75)" if best_score >= 0.75 else "Target not met")